# HireFlow 系统评估报告

In [1]:
# ================================================================
# Cell 1: 初始化 (路径修复 + 时间戳)
# ================================================================
import sys, os, time, json
from datetime import datetime
import pytz

# 修复路径: notebook 在 evaluation/ 子目录，app 在上级
project_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "evaluation" else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 时间戳 (悉尼时区)
sydney_tz = pytz.timezone("Australia/Sydney")
now = datetime.now(sydney_tz)
report_time = now.strftime("%I:%M %p")
report_date = now.strftime("%Y-%m-%d")

print(f"HireFlow 评估报告 | {report_date} {report_time}")

HireFlow 评估报告 | 2026-05-31 04:01 PM


In [2]:
# ================================================================
# Cell 2: 系统健康检查 (配置 + LLM + DB 连接)
# ================================================================
from app.utils.config import settings
from openai import OpenAI

results = {"llm": False, "embedding": False, "postgres": False, "qdrant": False}

# --- 配置 ---
print("【配置】")
print(f"  LLM: {settings.llm.mode} ({settings.llm.local_model if settings.llm.mode == 'local' else settings.llm.cloud_model})")
print(f"  Embedding: {settings.embedding.mode} ({settings.embedding.local_model})")
print(f"  DB: {settings.database.url.split('@')[1] if '@' in settings.database.url else settings.database.url}")
print(f"  Qdrant: {settings.qdrant.url}")

# --- LLM ---
print("\n【LLM 连接】")
try:
    client = OpenAI(base_url=settings.llm.local_base_url, api_key=settings.llm.local_api_key)
    models = [m.id for m in client.models.list().data]
    if settings.llm.local_model in models:
        print(f"  ✅ {settings.llm.local_model}")
        results["llm"] = True
    else:
        print(f"  ⚠️ 模型未找到, 可用: {models[:3]}")
except Exception as e:
    print(f"  ❌ {str(e)[:120]}")

# --- Embedding ---
print("\n【Embedding 连接】")
try:
    client = OpenAI(base_url=settings.embedding.local_base_url, api_key=settings.embedding.local_api_key)
    resp = client.embeddings.create(model=settings.embedding.local_model, input="test")
    dim = len(resp.data[0].embedding)
    print(f"  ✅ {settings.embedding.local_model} (维度={dim})")
    results["embedding"] = True
except Exception as e:
    print(f"  ❌ {str(e)[:120]}")

# --- PostgreSQL ---
print("\n【PostgreSQL 连接】")
try:
    from app.database.session import init_db, engine
    from sqlalchemy import inspect
    conn = engine.connect()
    conn.close()
    init_db()
    tables = inspect(engine).get_table_names()
    print(f"  ✅ 已连接, 表: {', '.join(tables)}")
    results["postgres"] = True
except Exception as e:
    print(f"  ❌ {str(e)[:120]}")
    print(f"  💡 docker compose up -d postgres")

# --- Qdrant ---
print("\n【Qdrant 连接】")
try:
    from qdrant_client import QdrantClient
    client = QdrantClient(url=settings.qdrant.url)
    cols = [c.name for c in client.get_collections().collections]
    print(f"  ✅ 已连接, 集合: {cols if cols else '(空)'}")
    results["qdrant"] = True
except Exception as e:
    print(f"  ❌ {str(e)[:120]}")
    print(f"  💡 docker compose up -d qdrant")

# --- 汇总 ---
all_ok = all(results.values())
print(f"\n健康检查: {'✅ 全部通过' if all_ok else '❌ 有问题: ' + ', '.join(k for k,v in results.items() if not v)}")

【配置】
  LLM: local (hermes-3-llama-3.1-8b)
  Embedding: local (text-embedding-qwen3-embedding-4b)
  DB: localhost:5432/hireflow
  Qdrant: http://localhost:6333

【LLM 连接】


  ✅ hermes-3-llama-3.1-8b

【Embedding 连接】


  ✅ text-embedding-qwen3-embedding-4b (维度=2560)

【PostgreSQL 连接】
  ✅ 已连接, 表: candidates, resume_chunks, jobs, match_results, interview_questions, interview_evaluations, email_drafts

【Qdrant 连接】


  ✅ 已连接, 集合: (空)

健康检查: ✅ 全部通过


In [3]:
# ================================================================
# Cell 3: Pipeline 运行测试 (直接用 await, 不用 asyncio.run)
# ================================================================
from app.agents.jd_agent import analyze_jd
from app.agents.resume_agent import batch_parse_resumes
from app.agents.match_agent import batch_match_candidates
from app.agents.ranking_agent import rank_candidates

print(f"Pipeline 测试开始 ({report_time})\n")

# --- 测试数据 ---
test_jd = """
岗位名称: Python 后端开发工程师
必备技能: Python, FastAPI, PostgreSQL, Docker, Git
加分技能: LangChain, RAG, Redis
岗位职责: 开发后端API, 数据库设计, 编写单元测试
学历要求: 计算机相关专业本科及以上
经验要求: 0-3年
"""

test_resumes = {
    "E001": "姓名: 张工\n技能: Python, FastAPI, PostgreSQL, Docker, Git, Redis\n项目: 电商API - FastAPI+PostgreSQL+Docker\n教育: 2020-2024 北大 CS学士\n经历: 2023某公司Python实习生",
    "E002": "姓名: 李工\n技能: Python, Django, MySQL, Docker, Git\n项目: 博客系统 - Django+MySQL\n教育: 2019-2023 浙大 SE学士\n经历: 2022某公司Django实习生",
    "E003": "姓名: 王工\n技能: Python, FastAPI, PostgreSQL, LangChain, RAG, Docker, Git, Redis\n项目: RAG问答系统 - FastAPI+LangChain+Qdrant\n教育: 2021-2023 清华 AI硕士\n经历: 2023某AI公司后端实习生",
}

timeline = []
total_start = time.time()

# Step 1: JD 解析
print("  [1/4] JD 解析...")
t0 = time.time()
jd_profile = await analyze_jd(test_jd)
t1 = time.time() - t0
timeline.append(("JD解析", t1))
print(f"        耗时: {t1:.1f}s 岗位: {jd_profile.get('job_title')}")

# Step 2: 简历解析
print("  [2/4] 简历解析...")
t0 = time.time()
profiles = await batch_parse_resumes(test_resumes)
t2 = time.time() - t0
timeline.append(("简历解析", t2))
for p in profiles:
    print(f"        {p.get('name')}: {len(p.get('skills',[]))}技能, {len(p.get('education',[]))}教育")
print(f"        耗时: {t2:.1f}s ({len(profiles)}份)")

# Step 3: 匹配评分
print("  [3/4] 匹配评分...")
ids = list(test_resumes.keys())
for i, p in enumerate(profiles):
    p["candidate_id"] = ids[i]
t0 = time.time()
rubric = jd_profile.pop("rubric", None)
matches = await batch_match_candidates(jd_profile, profiles, rubric=rubric)
t3 = time.time() - t0
timeline.append(("匹配评分", t3))
for m in matches:
    print(f"        {m.get('candidate_id')}: {m.get('total_score',0):.0f}分 → {m.get('recommendation','')}")
print(f"        耗时: {t3:.1f}s ({len(matches)}人)")

# Step 4: 排序
print("  [4/4] 排序...")
t0 = time.time()
ranking = await rank_candidates(matches)
t4 = time.time() - t0
timeline.append(("排序", t4))
print(f"        耗时: {t4:.1f}s")

total_time = time.time() - total_start
print(f"\n总耗时: {total_time:.1f}s\n")

Pipeline 测试开始 (04:01 PM)

  [1/4] JD 解析...


        耗时: 3.5s 岗位: Python 后端开发工程师
  [2/4] 简历解析...


        张工: 6技能, 1教育
        李工: 5技能, 1教育
        王工: 8技能, 1教育
        耗时: 18.2s (3份)
  [3/4] 匹配评分...


        E001: 0分 → Medium Match
        E002: 76分 → Medium Match
        E003: 0分 → Medium Match
        耗时: 24.8s (3人)
  [4/4] 排序...


        耗时: 2.0s

总耗时: 48.5s



In [4]:
# ================================================================
# Cell 4: 评估报告
# ================================================================
print("=" * 55)
print(f"  HireFlow 评估报告  |  {report_date}  {report_time}")
print("=" * 55)

# --- 性能时间线 ---
print("\n【性能: 各步骤耗时】")
for i, (name, t) in enumerate(timeline):
    pct = t / total_time * 100 if total_time > 0 else 0
    bar = "█" * max(1, int(pct / 3))
    print(f"  {i+1}. {name:8s} {t:5.1f}s  {bar}  {pct:.0f}%")
print(f"  {'─'*35}")
print(f"  总耗时: {total_time:.1f}s | 速度: {3/total_time:.2f}人/s" if total_time > 0 else "")

# --- 排序结果 ---
print("\n【排序结果】")
ranked = ranking.get("ranked_candidates", [])
header = f"  {'排名':<5} {'ID':<8} {'总分':<8} {'等级'}"
print(header)
print(f"  {'-'*42}")
for i, c in enumerate(ranked):
    score = c.get("total_score", 0)
    rec = c.get("recommendation", "")
    cid = c.get("candidate_id", "?")
    icon = "⭐" if score >= 80 else ("✅" if score >= 65 else ("⚠️" if score >= 50 else "❌"))
    print(f"  {icon} {i+1:<3}  {cid:<8} {score:<8.0f} {rec}")

# --- 维度分数详情 ---
print("\n【维度分数详情】")
for i, c in enumerate(ranked[:3]):
    score = c.get("total_score", 0)
    cid = c.get("candidate_id", "?")
    dims = c.get("dimension_scores", {})
    if not isinstance(dims, dict):
        continue
    identifier = cid if cid != "?" else f"第{i+1}名"
    print(f"\n  {identifier} ({score:.0f}分):")
    for dim_name, dim_score in dims.items():
        bar_len = max(1, int(abs(dim_score) / 30 * 12))
        bar = "█" * bar_len + "░" * (12 - bar_len)
        print(f"    {bar} {dim_name}: {dim_score}")

# --- 统计 ---
print("\n【统计摘要】")
summary = ranking.get("summary", {})
scores = [c.get("total_score", 0) for c in ranked]
print(f"  候选人: {len(scores)} | Strong:{summary.get('strong_match',0)} | Medium:{summary.get('medium_match',0)} | Weak:{summary.get('weak_match',0)} | NR:{summary.get('not_recommended',0)}")
if scores:
    print(f"  分数范围: {min(scores):.0f} - {max(scores):.0f} | 平均: {sum(scores)/len(scores):.0f} | 极差: {max(scores)-min(scores):.0f}")

# --- 问题检测 ---
print("\n【问题/建议】")
issues = []
if total_time > 120:
    issues.append(f"Pipeline 总耗时 {total_time:.0f}s > 120s，建议优化 LLM 调用并发")
if scores and max(scores) - min(scores) < 5:
    issues.append(f"候选人分数极差仅 {max(scores)-min(scores):.0f} 分，区分度不足")
if scores and sum(scores)/len(scores) < 50:
    issues.append(f"平均分 {sum(scores)/len(scores):.0f} 偏低，检查 JD 要求是否过高")
if settings.llm.mode == "local":
    issues.append("本地模型: 免费但慢 (每步 3-10s)，云端模式更快但需 API Key")
else:
    issues.append("云端模型: 速度快但按量计费")
if scores:
    slow_candidates = [c for c in ranked if c.get("total_score", 0) < 50]
    if len(slow_candidates) > len(ranked) / 2:
        issues.append(f"超过一半候选人低于 50 分，建议放宽 JD 要求或增加候选人池")
for issue in issues:
    print(f"  • {issue}")

print("\n" + "=" * 55)
print(f"  报告完成 ({report_time})")
print("=" * 55)

  HireFlow 评估报告  |  2026-05-31  04:01 PM

【性能: 各步骤耗时】
  1. JD解析       3.5s  ██  7%
  2. 简历解析      18.2s  ████████████  38%
  3. 匹配评分      24.8s  █████████████████  51%
  4. 排序         2.0s  █  4%
  ───────────────────────────────────
  总耗时: 48.5s | 速度: 0.06人/s

【排序结果】
  排名    ID       总分       等级
  ------------------------------------------
  ✅ 1    E002     76       Medium Match
  ❌ 2    E001     0        Not Recommended
  ❌ 3    E003     0        Not Recommended

【维度分数详情】

  E002 (76分):
    ██████████░░ technical_skills: 25.0
    ██████░░░░░░ project_relevance: 15.0
    ████░░░░░░░░ experience: 12.0
    ███░░░░░░░░░ education: 8.0
    ██░░░░░░░░░░ domain_relevance: 5.0
    █░░░░░░░░░░░ communication: 4.0
    █░░░░░░░░░░░ risk_penalty: -3.0

  E001 (0分):
    ██████████░░ technical_skills: 25.0
    ██████░░░░░░ project_relevance: 16.0
    ████░░░░░░░░ experience: 12.0
    ███░░░░░░░░░ education: 8.0
    ██░░░░░░░░░░ domain_relevance: 5.0
    █░░░░░░░░░░░ communication: 4.0
    █░░░░░░░

### 使用说明

**前置条件:**
```bash
conda activate hireflowagents
docker compose up -d postgres qdrant
```

**运行:**
```bash
cd evaluation && jupyter notebook 系统评估报告.ipynb
```

**注意:** 确保 LM Studio 已启动且 hermes-3-llama-3.1-8b 模型已加载。
如需用云端模型, 修改 `.env` 中 `LLM_MODE=cloud`。